
## Environment Setup

This notebook requires PyTorch and PyTorch Geometric (PyG). The required packages are defined in `environments/gpu.yml`.

### 1. Create the GPU environment

Open a Terminal in the downloaded repository's root directory.

For a new installation, run:

```bash
conda env create -f environments/gpu.yml
```

If the `t2d-gpu` environment already exists but PyTorch Geometric has not been installed, run:

```bash
conda run -n t2d-gpu python -m pip install torch-geometric==2.7.0
```

### 2. Register the Jupyter Kernel

```bash
conda run -n t2d-gpu python -m ipykernel install \
    --user \
    --name t2d-gpu \
    --display-name "T2D GPU"
```

### 3. Verify installation

```bash
conda run -n t2d-gpu python -c "import torch, torch_geometric; print(torch.__version__, torch_geometric.__version__)"
```

### 4. Select the Jupyter Kernel

Open this notebook and select:

**Kernel → Change Kernel → T2D GPU**

The GPU environment is shared with the DenseNet121 notebooks. No package installation is performed in the model-training cells.

Before executing this notebook, complete `2-1.Step2_DenseNet_ABD.ipynb` to generate `Outputs/Step2_DenseNet/ABD/ABD_ActivationVectors.csv`.


In [1]:
# ============================================================
# 0. GPU / PyTorch Geometric Environment Verification
# ============================================================

import sys
import torch
import torch_geometric

from torch_geometric.nn import GCNConv

print("=" * 65)
print("GNN ENVIRONMENT VERIFICATION")
print("=" * 65)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("CUDA runtime:", torch.version.cuda)

print("\nCUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Verify basic GCNConv operation
test_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

test_model = GCNConv(4, 4).to(test_device).eval()

test_x = torch.ones((2, 4), device=test_device)

test_edges = torch.tensor(
    [[0, 0, 1, 1],
     [0, 1, 0, 1]],
    dtype=torch.long,
    device=test_device
)

with torch.inference_mode():
    test_output = test_model(test_x, test_edges)

assert test_output.shape == (2, 4)

del test_model, test_x, test_edges, test_output

print("\nGNN ENVIRONMENT READY")

[LOCAL_PATH] TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GNN ENVIRONMENT VERIFICATION
Python: 3.11.18
PyTorch: 2.7.1+cu128
PyTorch Geometric: 2.7.0
CUDA runtime: 12.8

CUDA available: True
Visible GPUs: 2
GPU 0: NVIDIA A100-SXM4-40GB
GPU 1: NVIDIA A100-SXM4-40GB

GNN ENVIRONMENT READY


In [2]:
# ============================================================
# 1. User-defined Hyperparameters — ABD GNN
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score, accuracy_score

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GCNConv,
    global_add_pool,
    global_mean_pool
)
from monai.utils import set_determinism


# ============================================================
# 1.1 Project Configuration
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

SAMPLE_INFO_FILE = (
    PROJECT_ROOT
    / "Demo_data/1-1.Sample_information/sample_information.csv"
)

DENSENET_FEATURE_FILE = (
    PROJECT_ROOT
    / "Outputs/Step2_DenseNet/ABD/ABD_ActivationVectors.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "Outputs/Step3_GNN/ABD"


# ============================================================
# 1.2 Graph Construction
# ============================================================

NUM_NODE_FEATURES = 1024

DISTANCE_METRIC = "euclidean"

NODE_WEIGHT_THRESHOLD = 0.5

# Original implementation: all node pairs, including self-loops
INCLUDE_ALL_NODE_PAIRS = True


# ============================================================
# 1.3 Model Architecture
# ============================================================

MODEL_NAME = "GCN"

NUM_GCN_LAYERS = 3
HIDDEN_CHANNELS = 1024
NUM_CLASSES = 2

# Original implementation
POOLING_METHOD = "weighted_mean"

# Optional exploratory setting: "mean"


# ============================================================
# 1.4 Training Hyperparameters
# ============================================================

# Original study: 500 epochs
# GitHub demonstration: 20 epochs
MAX_EPOCHS = 20

LEARNING_RATE = 1e-5

OPTIMIZER = "Adam"
LOSS_FUNCTION = "CrossEntropyLoss"

TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1

VAL_INTERVAL = 1

GLOBAL_SEED = 0
MODEL_SEED = 12345


# ============================================================
# 1.5 Computational Configuration
# ============================================================

GPU_INDEX = 0
ALLOW_CPU_FALLBACK = True

NUM_WORKERS = 0


# ============================================================
# 1.6 Output Files
# ============================================================

CHECKPOINT_FILE = OUTPUT_DIR / "ABD_GNN_best_model.pth"
HISTORY_FILE = OUTPUT_DIR / "ABD_GNN_TrainingHistory.csv"
CONFIG_FILE = OUTPUT_DIR / "ABD_GNN_RunConfig.json"

GRAPH_SUMMARY_FILE = (
    OUTPUT_DIR / "ABD_GNN_GraphSummary.csv"
)

FEATURE_FILE = (
    OUTPUT_DIR / "ABD_GNN_ActivationVectors.csv"
)

PREDICTIONS_FILE = (
    OUTPUT_DIR / "ABD_GNN_Predictions.csv"
)


# ============================================================
# 1.7 Device Selection
# ============================================================

if torch.cuda.is_available():

    if GPU_INDEX >= torch.cuda.device_count():
        raise ValueError(
            f"GPU_INDEX={GPU_INDEX} is not available."
        )

    DEVICE = torch.device(f"cuda:{GPU_INDEX}")

else:

    if not ALLOW_CPU_FALLBACK:
        raise RuntimeError("CUDA is unavailable.")

    DEVICE = torch.device("cpu")


# ============================================================
# 1.8 Verification
# ============================================================

set_determinism(seed=GLOBAL_SEED)

assert DISTANCE_METRIC == "euclidean"
assert INCLUDE_ALL_NODE_PAIRS
assert NUM_GCN_LAYERS == 3
assert POOLING_METHOD in {"weighted_mean", "mean"}
assert MAX_EPOCHS >= 1
assert 1 <= VAL_INTERVAL <= MAX_EPOCHS

assert SAMPLE_INFO_FILE.is_file(), SAMPLE_INFO_FILE
assert DENSENET_FEATURE_FILE.is_file(), (
    "ABD DenseNet activation vectors not found."
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 65)
print("ABD GNN — HYPERPARAMETERS")
print("=" * 65)

print("Model:", MODEL_NAME)
print("GCN layers:", NUM_GCN_LAYERS)
print("Node feature dimensions:", NUM_NODE_FEATURES)
print("Hidden channels:", HIDDEN_CHANNELS)

print("\nGraph construction")
print("Distance:", DISTANCE_METRIC)
print("Node weight threshold:", NODE_WEIGHT_THRESHOLD)
print("All node pairs:", INCLUDE_ALL_NODE_PAIRS)
print("Readout:", POOLING_METHOD)

print("\nTraining")
print("Optimizer:", OPTIMIZER)
print("Learning rate:", LEARNING_RATE)
print("Maximum epochs:", MAX_EPOCHS)
print("Training batch size:", TRAIN_BATCH_SIZE)
print("Validation interval:", VAL_INTERVAL)

print("\nDevice:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))

print("\nConfiguration completed.")

ABD GNN — HYPERPARAMETERS
Model: GCN
GCN layers: 3
Node feature dimensions: 1024
Hidden channels: 1024

Graph construction
Distance: euclidean
Node weight threshold: 0.5
All node pairs: True
Readout: weighted_mean

Training
Optimizer: Adam
Learning rate: 1e-05
Maximum epochs: 20
Training batch size: 64
Validation interval: 1

Device: cuda:0
GPU: NVIDIA A100-SXM4-40GB

Configuration completed.


In [3]:
# ============================================================
# 2. Build Participant-level ABD Graphs
# ============================================================

# ============================================================
# 2.1 Load Sample Information
# ============================================================

info = pd.read_csv(
    SAMPLE_INFO_FILE,
    dtype={"Demo_ID": str}
)

features = pd.read_csv(
    DENSENET_FEATURE_FILE,
    dtype={"Demo_ID": str}
)

expected_splits = {
    "training1": 32,
    "training2": 32,
    "validation": 16,
    "testing": 20,
}

assert len(info) == 100
assert info["Demo_ID"].is_unique

assert (
    info["Dataset"].value_counts().to_dict()
    == expected_splits
)


# ============================================================
# 2.2 Validate Activation Vectors
# ============================================================

AV_COLUMNS = [
    f"AV_{i:04d}"
    for i in range(NUM_NODE_FEATURES)
]

required_columns = {
    "Demo_ID",
    "Dataset",
    "T2D",
    "Image_file",
    *AV_COLUMNS,
}

missing = required_columns - set(features.columns)

assert not missing, f"Missing columns: {missing}"

assert len(features) == int(info["N_ABD"].sum())
assert features["Demo_ID"].nunique() == 100
assert features["Image_file"].is_unique

assert np.isfinite(
    features[AV_COLUMNS].to_numpy(dtype=np.float32)
).all()

# Verify participant-level partition consistency
info_idx = info.set_index("Demo_ID")

assert (
    features["Dataset"]
    == features["Demo_ID"].map(info_idx["Dataset"])
).all()

assert (
    pd.to_numeric(features["T2D"])
    == features["Demo_ID"].map(info_idx["T2D"])
).all()


# ============================================================
# 2.3 Participant Graph Construction
# ============================================================

def build_graph(participant_df):

    participant_df = (
        participant_df
        .sort_values("Image_file")
        .reset_index(drop=True)
    )

    demo_id = str(participant_df["Demo_ID"].iloc[0])
    split = str(participant_df["Dataset"].iloc[0])
    label = int(participant_df["T2D"].iloc[0])

    vectors = participant_df[
        AV_COLUMNS
    ].to_numpy(dtype=np.float32)

    n_nodes = len(vectors)

    # --------------------------------------------------------
    # Euclidean distance
    # --------------------------------------------------------

    distances = cdist(
        vectors,
        vectors,
        metric=DISTANCE_METRIC
    )

    # Original edge weight
    weights = 1.0 / (
        distances + (distances == 0)
    )

    # --------------------------------------------------------
    # Fully connected directed graph, including self-loops
    # --------------------------------------------------------

    src, dst = np.meshgrid(
        np.arange(n_nodes),
        np.arange(n_nodes),
        indexing="ij"
    )

    edge_index = torch.tensor(
        np.stack([
            src.ravel(),
            dst.ravel()
        ]),
        dtype=torch.long
    )

    edge_weight = torch.tensor(
        weights.ravel(),
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # Node weights
    # --------------------------------------------------------

    node_weights = (
        (weights > NODE_WEIGHT_THRESHOLD)
        .sum(axis=1)
        .astype(np.float32)
    )

    sum_node_weights = float(node_weights.sum())

    assert sum_node_weights > 0

    # --------------------------------------------------------
    # PyTorch Geometric Data
    # --------------------------------------------------------

    graph = Data(

        x=torch.from_numpy(vectors),

        edge_index=edge_index,

        edge_weight=edge_weight,

        img_weights=torch.from_numpy(
            node_weights[:, None]
        ),

        sum_img_weights=torch.tensor(
            [sum_node_weights],
            dtype=torch.float32
        ),

        y=torch.tensor(
            [label],
            dtype=torch.long
        ),

        demo_id=demo_id,

    )

    summary = {
        "Demo_ID": demo_id,
        "Dataset": split,
        "T2D": label,
        "N_images": n_nodes,
        "N_nodes": n_nodes,
        "N_edges": int(edge_index.shape[1]),
        "Sum_node_weight": sum_node_weights,
    }

    return graph, summary


# ============================================================
# 2.4 Construct All 100 Graphs
# ============================================================

splits = list(expected_splits.keys())

graphs_by_split = {
    split: []
    for split in splits
}

summary_records = []

for demo_id, group in features.groupby("Demo_ID"):

    graph, summary = build_graph(group)

    assert summary["N_edges"] == (
        summary["N_nodes"] ** 2
    )

    graphs_by_split[
        summary["Dataset"]
    ].append(graph)

    summary_records.append(summary)


graph_summary = pd.DataFrame(summary_records)

assert len(graph_summary) == 100

assert (
    graph_summary["Dataset"].value_counts().to_dict()
    == expected_splits
)

assert graph_summary["N_nodes"].sum() == len(features)

graph_summary.to_csv(
    GRAPH_SUMMARY_FILE,
    index=False
)


# ============================================================
# 2.5 Construct DataLoaders
# ============================================================

def make_loader(split, batch_size, shuffle=False):

    return DataLoader(

        graphs_by_split[split],

        batch_size=batch_size,

        shuffle=shuffle,

        num_workers=NUM_WORKERS,

    )


train_loader = make_loader(
    "training2",
    TRAIN_BATCH_SIZE,
    shuffle=True
)

val_loader = make_loader(
    "validation",
    EVAL_BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 2.6 Verify Graph DataLoader
# ============================================================

test_graph = next(iter(val_loader)).to(DEVICE)

assert test_graph.x.shape[1] == NUM_NODE_FEATURES

assert test_graph.edge_index.shape[0] == 2

assert test_graph.sum_img_weights.numel() == (
    test_graph.num_graphs
)


print("=" * 65)
print("ABD GRAPH CONSTRUCTION")
print("=" * 65)

print("Total graphs:", len(graph_summary))
print("Total nodes:", graph_summary["N_nodes"].sum())
print("Total edges:", graph_summary["N_edges"].sum())

display(
    graph_summary.groupby("Dataset")
    .agg(
        Graphs=("Demo_ID", "count"),
        Images=("N_images", "sum"),
        Edges=("N_edges", "sum")
    )
    .reindex(splits)
)

print("\nGraph DataLoader ready.")

ABD GRAPH CONSTRUCTION
Total graphs: 100
Total nodes: 2487
Total edges: 64501


,Graphs,Images,Edges
Dataset,,,
training1,32,784,20284
training2,32,786,19784
validation,16,385,9491
testing,20,532,14942



Graph DataLoader ready.


In [4]:
# ============================================================
# 3. Train ABD Graph Convolutional Network
# ============================================================

# ============================================================
# 3.1 Define GNN Architecture
# ============================================================

class ABD_GCN(torch.nn.Module):

    def __init__(
        self,
        in_features,
        hidden,
        out_classes
    ):

        super().__init__()

        self.convs = torch.nn.ModuleList([

            GCNConv(
                in_features if i == 0 else hidden,
                hidden
            )

            for i in range(NUM_GCN_LAYERS)

        ])

        self.classifier = torch.nn.Linear(
            hidden,
            out_classes
        )

    def forward(self, batch):

        x = batch.x

        # Three graph convolutional layers
        for i, conv in enumerate(self.convs):

            x = conv(
                x,
                batch.edge_index,
                batch.edge_weight
            )

            if i < len(self.convs) - 1:
                x = F.relu(x)

        # Participant-level graph readout
        if POOLING_METHOD == "weighted_mean":

            weighted_sum = global_add_pool(
                x * batch.img_weights,
                batch.batch
            )

            embedding = (
                weighted_sum
                / batch.sum_img_weights.view(-1, 1)
            )

        elif POOLING_METHOD == "mean":

            embedding = global_mean_pool(
                x,
                batch.batch
            )

        else:
            raise ValueError(POOLING_METHOD)

        logits = self.classifier(embedding)

        return logits, embedding


# ============================================================
# 3.2 Initialize Model
# ============================================================

torch.manual_seed(MODEL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_SEED)

model = ABD_GCN(
    in_features=NUM_NODE_FEATURES,
    hidden=HIDDEN_CHANNELS,
    out_classes=NUM_CLASSES
).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 3.3 Save Run Configuration
# ============================================================

run_config = {

    "Model": MODEL_NAME,

    "Input_features": NUM_NODE_FEATURES,
    "GCN_layers": NUM_GCN_LAYERS,
    "Hidden_channels": HIDDEN_CHANNELS,

    "Distance_metric": DISTANCE_METRIC,
    "Node_weight_threshold": NODE_WEIGHT_THRESHOLD,

    "Fully_connected_graph": INCLUDE_ALL_NODE_PAIRS,
    "Graph_readout": POOLING_METHOD,

    "Optimizer": OPTIMIZER,
    "Learning_rate": LEARNING_RATE,
    "Loss_function": LOSS_FUNCTION,

    "Max_epochs_original": 500,
    "Max_epochs_demo": MAX_EPOCHS,

    "Training_batch_size": TRAIN_BATCH_SIZE,
    "Validation_interval": VAL_INTERVAL,

    "Training_partition": "training2",
    "Selection_partition": "validation",

    "Global_seed": GLOBAL_SEED,
    "Model_seed": MODEL_SEED,

    "Device": str(DEVICE),

    "PyTorch": torch.__version__,
    "PyG": torch_geometric.__version__,

}

CONFIG_FILE.write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8"
)


# ============================================================
# 3.4 Validation Function
# ============================================================

@torch.no_grad()
def evaluate_gnn(current_model, loader):

    current_model.eval()

    true_labels = []
    probabilities = []
    total_loss = 0.0
    total_n = 0

    for batch in loader:

        batch = batch.to(DEVICE)

        logits, _ = current_model(batch)

        loss = criterion(logits, batch.y)

        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        true_labels.extend(
            batch.y.cpu().tolist()
        )

        probabilities.extend(
            probs.cpu().tolist()
        )

        total_loss += (
            float(loss.item()) * batch.num_graphs
        )

        total_n += batch.num_graphs

    if len(set(true_labels)) != 2:
        raise ValueError(
            "Validation set must contain both classes."
        )

    return {

        "AUC": roc_auc_score(
            true_labels,
            probabilities
        ),

        "Accuracy": accuracy_score(
            true_labels,
            np.asarray(probabilities) >= 0.5
        ),

        "Loss": total_loss / total_n,

    }


# ============================================================
# 3.5 Train GNN
# ============================================================

history = []

best_auc = float("-inf")
best_epoch = None

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)


for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.perf_counter()

    model.train()

    total_train_loss = 0.0
    total_train_graphs = 0

    for batch in train_loader:

        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = model(batch)

        loss = criterion(
            logits,
            batch.y
        )

        loss.backward()

        optimizer.step()

        total_train_loss += (
            float(loss.item()) * batch.num_graphs
        )

        total_train_graphs += batch.num_graphs

    train_loss = (
        total_train_loss / total_train_graphs
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_auc = np.nan
    val_accuracy = np.nan
    val_loss = np.nan

    if epoch % VAL_INTERVAL == 0:

        val_result = evaluate_gnn(
            model,
            val_loader
        )

        val_auc = float(val_result["AUC"])
        val_accuracy = float(val_result["Accuracy"])
        val_loss = float(val_result["Loss"])

        if val_auc > best_auc:

            best_auc = val_auc
            best_epoch = epoch

            torch.save(
                model.state_dict(),
                CHECKPOINT_FILE
            )


    # --------------------------------------------------------
    # Save History
    # --------------------------------------------------------

    epoch_seconds = (
        time.perf_counter() - start_time
    )

    history.append({

        "Epoch": epoch,
        "Training_loss": train_loss,
        "Validation_loss": val_loss,
        "Validation_AUC": val_auc,
        "Validation_accuracy": val_accuracy,
        "Epoch_seconds": epoch_seconds,

    })

    pd.DataFrame(history).to_csv(
        HISTORY_FILE,
        index=False
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Loss={train_loss:.4f} | "
        f"Val AUC={val_auc:.4f} | "
        f"Best AUC={best_auc:.4f} | "
        f"Time={epoch_seconds:.1f}s",
        flush=True
    )


# ============================================================
# 3.6 Training Summary
# ============================================================

assert CHECKPOINT_FILE.is_file()

print("\n" + "=" * 65)
print("ABD GNN TRAINING COMPLETED")
print("=" * 65)

print("Best epoch:", best_epoch)
print("Best validation AUC:", round(best_auc, 4))

if DEVICE.type == "cuda":

    peak_memory = (
        torch.cuda.max_memory_allocated(DEVICE) / 1024**3
    )

    print(
        "Peak allocated GPU memory (GiB):",
        round(peak_memory, 3)
    )

print(
    "\nCheckpoint:",
    CHECKPOINT_FILE.relative_to(PROJECT_ROOT)
)

Epoch 01/20 | Loss=0.6937 | Val AUC=0.2656 | Best AUC=0.2656 | Time=0.5s
Epoch 02/20 | Loss=0.6928 | Val AUC=0.2969 | Best AUC=0.2969 | Time=0.1s
Epoch 03/20 | Loss=0.6922 | Val AUC=0.3281 | Best AUC=0.3281 | Time=0.1s
Epoch 04/20 | Loss=0.6918 | Val AUC=0.3594 | Best AUC=0.3594 | Time=0.1s
Epoch 05/20 | Loss=0.6914 | Val AUC=0.3594 | Best AUC=0.3594 | Time=0.1s
Epoch 06/20 | Loss=0.6912 | Val AUC=0.3906 | Best AUC=0.3906 | Time=0.1s
Epoch 07/20 | Loss=0.6908 | Val AUC=0.3906 | Best AUC=0.3906 | Time=0.1s
Epoch 08/20 | Loss=0.6905 | Val AUC=0.3906 | Best AUC=0.3906 | Time=0.1s
Epoch 09/20 | Loss=0.6902 | Val AUC=0.4062 | Best AUC=0.4062 | Time=0.1s
Epoch 10/20 | Loss=0.6898 | Val AUC=0.4062 | Best AUC=0.4062 | Time=0.1s
Epoch 11/20 | Loss=0.6895 | Val AUC=0.3906 | Best AUC=0.4062 | Time=0.1s
Epoch 12/20 | Loss=0.6891 | Val AUC=0.4062 | Best AUC=0.4062 | Time=0.1s
Epoch 13/20 | Loss=0.6888 | Val AUC=0.4062 | Best AUC=0.4062 | Time=0.1s
Epoch 14/20 | Loss=0.6885 | Val AUC=0.4062 | Best A

In [5]:
# ============================================================
# 4. Extract Participant-level GNN Activation Vectors
# ============================================================

# ============================================================
# 4.1 Load Best Model
# ============================================================

assert CHECKPOINT_FILE.is_file(), CHECKPOINT_FILE

model.load_state_dict(
    torch.load(
        CHECKPOINT_FILE,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()


# ============================================================
# 4.2 Extract GNN Features
# ============================================================

feature_records = []
prediction_records = []

with torch.no_grad():

    for split in splits:

        loader = make_loader(
            split,
            EVAL_BATCH_SIZE,
            shuffle=False
        )

        for batch in loader:

            batch = batch.to(DEVICE)

            logits, embedding = model(batch)

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1].cpu().numpy()

            vectors = embedding.cpu().numpy()

            labels = batch.y.cpu().numpy()

            demo_ids = batch.demo_id

            if isinstance(demo_ids, str):
                demo_ids = [demo_ids]

            assert len(demo_ids) == len(vectors)

            for i, demo_id in enumerate(demo_ids):

                base = {
                    "Demo_ID": str(demo_id),
                    "Dataset": split,
                    "T2D": int(labels[i]),
                }

                prediction_records.append({

                    **base,

                    "Predicted_Probability": float(
                        probs[i]
                    ),

                })

                feature_records.append({

                    **base,

                    **{
                        f"GNN_{j:04d}": float(value)

                        for j, value in enumerate(
                            vectors[i]
                        )
                    },

                })


# ============================================================
# 4.3 Create Output DataFrames
# ============================================================

feature_df = pd.DataFrame(
    feature_records
).sort_values("Demo_ID").reset_index(drop=True)

prediction_df = pd.DataFrame(
    prediction_records
).sort_values("Demo_ID").reset_index(drop=True)

GNN_COLUMNS = [
    f"GNN_{i:04d}"
    for i in range(HIDDEN_CHANNELS)
]


# ============================================================
# 4.4 Validate Outputs
# ============================================================

assert len(feature_df) == 100
assert len(prediction_df) == 100

assert feature_df["Demo_ID"].is_unique
assert prediction_df["Demo_ID"].is_unique

assert len(GNN_COLUMNS) == HIDDEN_CHANNELS

assert np.isfinite(
    feature_df[GNN_COLUMNS].to_numpy()
).all()

assert (
    feature_df["Dataset"].value_counts().to_dict()
    == expected_splits
)

assert prediction_df[
    "Predicted_Probability"
].between(0, 1).all()


# ============================================================
# 4.5 Save Outputs
# ============================================================

feature_df.to_csv(
    FEATURE_FILE,
    index=False
)

prediction_df.to_csv(
    PREDICTIONS_FILE,
    index=False
)


# ============================================================
# 4.6 Summary
# ============================================================

print("=" * 65)
print("ABD GNN ACTIVATION VECTOR EXTRACTION")
print("=" * 65)

print("Participants:", len(feature_df))
print("GNN feature dimensions:", len(GNN_COLUMNS))

print("\nParticipants by dataset:")

print(
    feature_df["Dataset"]
    .value_counts()
    .reindex(splits)
    .to_string()
)

print("\nOutput files:")

for output_file in [
    CHECKPOINT_FILE,
    HISTORY_FILE,
    CONFIG_FILE,
    GRAPH_SUMMARY_FILE,
    FEATURE_FILE,
    PREDICTIONS_FILE,
]:
    print(output_file.relative_to(PROJECT_ROOT))


display(
    feature_df[
        [
            "Demo_ID",
            "Dataset",
            "GNN_0000",
            "GNN_0001",
            "GNN_0002",
        ]
    ].head()
)

print("\nGNN FEATURE EXTRACTION COMPLETED")

ABD GNN ACTIVATION VECTOR EXTRACTION
Participants: 100
GNN feature dimensions: 1024

Participants by dataset:
Dataset
training1     32
training2     32
validation    16
testing       20

Output files:
Outputs/Step3_GNN/ABD/ABD_GNN_best_model.pth
Outputs/Step3_GNN/ABD/ABD_GNN_TrainingHistory.csv
Outputs/Step3_GNN/ABD/ABD_GNN_RunConfig.json
Outputs/Step3_GNN/ABD/ABD_GNN_GraphSummary.csv
Outputs/Step3_GNN/ABD/ABD_GNN_ActivationVectors.csv
Outputs/Step3_GNN/ABD/ABD_GNN_Predictions.csv


,Demo_ID,Dataset,GNN_0000,GNN_0001,GNN_0002
0,D001,training2,-0.011798,0.133992,0.167634
1,D002,testing,-0.011806,0.120093,0.163269
2,D003,training2,-0.004990,0.111255,0.166192
3,D004,training1,-0.006427,0.124058,0.164860
4,D005,training2,-0.007820,0.132767,0.167151



GNN FEATURE EXTRACTION COMPLETED
